In [ ]:
import numpy as np
from glob import glob
import os.path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Conv2D, concatenate
import sys
sys.path.append('../')
from keras_vision_transformer import swin_layers
from keras_vision_transformer import transformer_layers
from keras_vision_transformer import utils
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def swin_transformer_stack(X, stack_num, embed_dim, num_patch, num_heads, window_size, num_mlp, shift_window=True, name=''):
    '''
    Stacked Swin Transformers that share the same token size.

    Alternated Window-MSA and Swin-MSA will be configured if `shift_window=True`, Window-MSA only otherwise.
    *Dropout is turned off.
    '''
    # Turn-off dropouts
    mlp_drop_rate = 0 # Droupout after each MLP layer
    attn_drop_rate = 0 # Dropout after Swin-Attention
    proj_drop_rate = 0 # Dropout at the end of each Swin-Attention block, i.e., after linear projections
    drop_path_rate = 0 # Drop-path within skip-connections

    qkv_bias = True # Convert embedded patches to query, key, and values with a learnable additive value
    qk_scale = None # None: Re-scale query based on embed dimensions per attention head # Float for user specified scaling factor

    if shift_window:
        shift_size = window_size // 2
    else:
        shift_size = 0

    for i in range(stack_num):

        if i % 2 == 0:
            shift_size_temp = 0
        else:
            shift_size_temp = shift_size

        X = swin_layers.SwinTransformerBlock(dim=embed_dim,
                                             num_patch=num_patch,
                                             num_heads=num_heads,
                                             window_size=window_size,
                                             shift_size=shift_size_temp,
                                             num_mlp=num_mlp,
                                             qkv_bias=qkv_bias,
                                             qk_scale=qk_scale,
                                             mlp_drop=mlp_drop_rate,
                                             attn_drop=attn_drop_rate,
                                             proj_drop=proj_drop_rate,
                                             drop_path_prob=drop_path_rate,
                                             name='name{}'.format(i))(X)
    return X


def swin_unet_2d_base(input_tensor, filter_num_begin, depth, stack_num_down, stack_num_up,
                      patch_size, num_heads, window_size, num_mlp, shift_window=True, name='swin_unet'):
    '''
    The base of Swin-UNET.

    The general structure:

    1. Input image --> a sequence of patches --> tokenize these patches
    2. Downsampling: swin-transformer --> patch merging (pooling)
    3. Upsampling: concatenate --> swin-transfprmer --> patch expanding (unpooling)
    4. Model head

    '''
    # Compute number be patches to be embeded
    input_size = input_tensor.shape.as_list()[1:]
    num_patch_x = input_size[0]//patch_size[0]
    num_patch_y = input_size[1]//patch_size[1]

    # Number of Embedded dimensions
    embed_dim = filter_num_begin

    depth_ = depth

    X_skip = []

    X = input_tensor

    # Patch extraction
    X = transformer_layers.patch_extract(patch_size)(X)

    # Embed patches to tokens
    X = transformer_layers.patch_embedding(num_patch_x*num_patch_y, embed_dim)(X)

    # The first Swin Transformer stack
    X = swin_transformer_stack(X,
                               stack_num=stack_num_down,
                               embed_dim=embed_dim,
                               num_patch=(num_patch_x, num_patch_y),
                               num_heads=num_heads[0],
                               window_size=window_size[0],
                               num_mlp=num_mlp,
                               shift_window=shift_window,
                               name='{}_swin_down0'.format(name))
    X_skip.append(X)

    # Downsampling blocks
    for i in range(depth_-1):

        # Patch merging
        X = transformer_layers.patch_merging((num_patch_x, num_patch_y), embed_dim=embed_dim, name='down{}'.format(i))(X)

        # update token shape info
        embed_dim = embed_dim*2
        num_patch_x = num_patch_x//2
        num_patch_y = num_patch_y//2

        # Swin Transformer stacks
        X = swin_transformer_stack(X,
                                   stack_num=stack_num_down,
                                   embed_dim=embed_dim,
                                   num_patch=(num_patch_x, num_patch_y),
                                   num_heads=num_heads[i+1],
                                   window_size=window_size[i+1],
                                   num_mlp=num_mlp,
                                   shift_window=shift_window,
                                   name='{}_swin_down{}'.format(name, i+1))

        # Store tensors for concat
        X_skip.append(X)

    # reverse indexing encoded tensors and hyperparams
    X_skip = X_skip[::-1]
    num_heads = num_heads[::-1]
    window_size = window_size[::-1]

    # upsampling begins at the deepest available tensor
    X = X_skip[0]

    # other tensors are preserved for concatenation
    X_decode = X_skip[1:]

    depth_decode = len(X_decode)

    for i in range(depth_decode):

        # Patch expanding
        X = transformer_layers.patch_expanding(num_patch=(num_patch_x, num_patch_y),
                                               embed_dim=embed_dim,
                                               upsample_rate=2,
                                               return_vector=True)(X)


        # update token shape info
        embed_dim = embed_dim//2
        num_patch_x = num_patch_x*2
        num_patch_y = num_patch_y*2

        # Concatenation and linear projection
        X = concatenate([X, X_decode[i]], axis=-1, name='{}_concat_{}'.format(name, i))
        X = Dense(embed_dim, use_bias=False, name='{}_concat_linear_proj_{}'.format(name, i))(X)

        # Swin Transformer stacks
        X = swin_transformer_stack(X,
                                   stack_num=stack_num_up,
                                   embed_dim=embed_dim,
                                   num_patch=(num_patch_x, num_patch_y),
                                   num_heads=num_heads[i],
                                   window_size=window_size[i],
                                   num_mlp=num_mlp,
                                   shift_window=shift_window,
                                   name='{}_swin_up{}'.format(name, i))

    # The last expanding layer; it produces full-size feature maps based on the patch size
    # !!! <--- "patch_size[0]" is used; it assumes patch_size = (size, size)

    X = transformer_layers.patch_expanding(num_patch=(num_patch_x, num_patch_y),
                                           embed_dim=embed_dim,
                                           upsample_rate=patch_size[0],
                                           return_vector=False)(X)

    return X
def input_data_process(input_array):
    '''converting pixel vales to [0, 1]'''
    return input_array

def target_data_process(target_array):
    '''Converting tri-mask of {1, 2, 3} to three categories.'''
    return target_array
def ax_decorate_box(ax):
    [j.set_linewidth(0) for j in ax.spines.values()]
    ax.tick_params(axis="both", which="both", bottom=False, top=False,
                   labelbottom=False, left=False, right=False, labelleft=False)
    return ax

In [ ]:
filter_num_begin_1 = 32     # number of channels in the first downsampling block; it is also the number of embedded dimensions
depth_1 = 4                  # the depth of SwinUNET; depth=4 means three down/upsampling levels and a bottom level
stack_num_down_1 = 2         # number of Swin Transformers per downsampling level
stack_num_up_1 = 2           # number of Swin Transformers per upsampling level
patch_size_1 = (4, 4)        # Extract 4-by-4 patches from the input image. Height and width of the patch must be equal.
num_heads_1 = [4, 8, 8, 8]   # number of attention heads per down/upsampling level
window_size_1 = [4, 2, 2, 2] # the size of attention window per down/upsampling level
num_mlp_1 = 512              # number of MLP nodes within the Transformer
shift_window=True          # Apply window shifting, i.e., Swin-MSA

In [ ]:
filter_num_begin = 32     # number of channels in the first downsampling block; it is also the number of embedded dimensions
depth = 4                  # the depth of SwinUNET; depth=4 means three down/upsampling levels and a bottom level
stack_num_down = 2         # number of Swin Transformers per downsampling level
stack_num_up = 2           # number of Swin Transformers per upsampling level
patch_size = (4, 4)        # Extract 4-by-4 patches from the input image. Height and width of the patch must be equal.
num_heads = [4, 8, 8, 8]   # number of attention heads per down/upsampling level
window_size = [4, 2, 2, 2] # the size of attention window per down/upsampling level
num_mlp = 512              # number of MLP nodes within the Transformer
shift_window=True          # Apply window shifting, i.e., Swin-MSA

In [ ]:
from __future__ import division
from keras.models import Model
from keras.layers import Input, concatenate, Conv2D, MaxPooling2D, UpSampling2D, Reshape, Dropout
from tensorflow.keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint, LearningRateScheduler
from keras import backend as K
# from keras.utils.vis_utils import plot_model as plot
from tensorflow.keras.optimizers import SGD
from keras.optimizers import *
from keras.layers import *
from tensorflow import matmul, math, cast, float32

from keras.backend import softmax
import math
from keras.initializers import HeUniform
# Input section

input_size = (256,256,3)
IN = Input(input_size)
N = input_size[0]

conv1 = SeparableConv2D(filters = 24, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(IN)
conv1 = BatchNormalization()(conv1)
conv1 = SeparableConv2D(filters = 24, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(conv1)
conv1 = BatchNormalization()(conv1)
pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

pool1 = swin_unet_2d_base(pool1, filter_num_begin, depth, stack_num_down, stack_num_up,
                      patch_size, num_heads, window_size, num_mlp,
                      shift_window=shift_window, name='swin_unet_E1')

conv2 = SeparableConv2D(filters = 48, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(pool1)
conv2 = BatchNormalization()(conv2)
conv2 = SeparableConv2D(filters = 48, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(conv2)
conv2 = BatchNormalization()(conv2)
pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

pool2 = swin_unet_2d_base(pool2, filter_num_begin, depth, stack_num_down, stack_num_up,
                      patch_size, num_heads, window_size, num_mlp,
                      shift_window=shift_window, name='swin_unet_E2')

conv3 = SeparableConv2D(filters = 96, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(pool2)
conv3 = BatchNormalization()(conv3)
conv3 = SeparableConv2D(filters = 96, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(conv3)
conv3 = BatchNormalization()(conv3)
drop3 = Dropout(0.5)(conv3)
pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)



# D1

conv4 = SeparableConv2D(filters = 192, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(pool3)
conv4 = BatchNormalization()(conv4)
conv4 = SeparableConv2D(filters = 192, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(conv4)
conv4_1 = BatchNormalization()(conv4)
drop4_1 = Dropout(0.5)(conv4_1)



# D2

conv4_2 = SeparableConv2D(filters = 192, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(drop4_1)
conv4_2 = BatchNormalization()(conv4_2)
conv4_2 = SeparableConv2D(filters = 192, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(conv4_2)
conv4_2 = BatchNormalization()(conv4_2)
drop4_2 = Dropout(0.5)(conv4_2)

# D3
merge_dense = concatenate([conv4_2,drop4_1], axis = 3)


conv4_3 = SeparableConv2D(filters = 192, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(merge_dense)
conv4_3 = SeparableConv2D(filters = 192, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(conv4_3)
drop4_3 = Dropout(0.5)(conv4_3)




up6 = Conv2DTranspose(96, kernel_size=2, strides=2, padding='same',kernel_initializer = 'he_normal')(drop4_3)
up6 = BatchNormalization(axis=3)(up6)
up6 = Activation('relu')(up6)

x1 = Reshape(target_shape=(1, np.int32(N/4), np.int32(N/4), 96))(drop3)
x2 = Reshape(target_shape=(1, np.int32(N/4), np.int32(N/4), 96))(up6)
merge6  = concatenate([x1,x2], axis = 1)
merge6 = ConvLSTM2D(filters = 384, kernel_size=(3, 3), padding='same', return_sequences = False, go_backwards = True,kernel_initializer = 'he_normal' )(merge6)

# X = convformer(merge6, 384)
X = swin_unet_2d_base(merge6, filter_num_begin, depth, stack_num_down, stack_num_up,
                      patch_size, num_heads, window_size, num_mlp,
                      shift_window=shift_window, name='swin_unet')

conv6 = SeparableConv2D(filters = 48, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(X)
conv6 = SeparableConv2D(filters = 48, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(conv6)

up7 = Conv2DTranspose(48, kernel_size=2, strides=2, padding='same',kernel_initializer = 'he_normal')(conv6)
up7 = BatchNormalization(axis=3)(up7)
up7 = Activation('relu')(up7)

x1 = Reshape(target_shape=(1, np.int32(N/2), np.int32(N/2), 48))(conv2)
x2 = Reshape(target_shape=(1, np.int32(N/2), np.int32(N/2), 48))(up7)
merge7  = concatenate([x1,x2], axis = 1)
merge7 = ConvLSTM2D(filters = 96, kernel_size=(3, 3), padding='same', return_sequences = False, go_backwards = True,kernel_initializer = 'he_normal' )(merge7)

# X1 = convformer(merge7, 96)
X1 = swin_unet_2d_base(merge7, filter_num_begin_1, depth_1, stack_num_down_1, stack_num_up_1,
                      patch_size_1, num_heads_1, window_size_1, num_mlp_1,
                      shift_window=shift_window, name='swin_unet')


conv7 = SeparableConv2D(filters = 48, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(X1)
conv7 = SeparableConv2D(filters = 48, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(conv7)

up8 = Conv2DTranspose(24, kernel_size=2, strides=2, padding='same',kernel_initializer = 'he_normal')(conv7)
up8 = BatchNormalization(axis=3)(up8)
up8 = Activation('relu')(up8)

x1 = Reshape(target_shape=(1, N, N, 24))(conv1)
x2 = Reshape(target_shape=(1, N, N, 24))(up8)
merge8  = concatenate([x1,x2], axis = 1)
merge8 = ConvLSTM2D(filters = 48, kernel_size=(3, 3), padding='same', return_sequences = False, go_backwards = True,kernel_initializer = 'he_normal' )(merge8)

# X2 = convformer(merge8, 48)
X2 = swin_unet_2d_base(merge8, filter_num_begin_1, depth_1, stack_num_down_1, stack_num_up_1,
                      patch_size_1, num_heads_1, window_size_1, num_mlp_1,
                      shift_window=shift_window, name='swin_unet')


conv8 = SeparableConv2D(filters = 24, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(X2)
conv8 = SeparableConv2D(filters = 24, kernel_size = (3, 3), activation = 'relu', kernel_initializer='he_normal', padding="same")(conv8)
conv8 = Conv2D(2, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv8)
conv9 = Conv2D(1, 1, activation = 'sigmoid')(conv8)

model = Model(inputs = [IN,], outputs = [conv9,])
model.compile(optimizer = Adam(lr = 1e-4), loss = DiceLoss, metrics = ['accuracy'])

In [ ]:
model.summary()

In [ ]:
from zipfile import ZipFile

filename = "/content/gdrive/MyDrive/ISIC_2017_Swin/test ISIC2018.zip"
with ZipFile(filename, 'r') as zip:
  zip.extractall()
  print('Done')

In [ ]:
train_data = '/content/train' #data path
valid_data = '/content/test'

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("ggplot")
%matplotlib inline
!pip install Keras-Preprocessing
from tqdm import tqdm_notebook, tnrange
from itertools import chain
from skimage.io import imread, imshow, concatenate_images
from skimage.transform import resize
from skimage.morphology import label
from sklearn.model_selection import train_test_split

import tensorflow as tf

from keras.models import Model, load_model
from keras.layers import Input, BatchNormalization, Activation, Dense, Dropout
from keras.layers.core import Lambda, RepeatVector, Reshape
from keras.layers.convolutional import Conv2D, Conv2DTranspose
from keras.layers.pooling import AveragePooling2D,MaxPooling2D, GlobalMaxPool2D
from keras.layers import Concatenate, add
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# from tensorflow.keras.optimizers import Adam
from keras_preprocessing.image import ImageDataGenerator, array_to_img, img_to_array, load_img
import warnings
warnings.filterwarnings("ignore")
random.seed(42)
import cv2

In [ ]:

import os
import numpy as np
im_height=256
im_width=256
# Get and resize train images and masks
def get_data(path):
    images_paths = os.path.join(path,'x')
    masks_paths = os.path.join(path,'y')

    # images_ids = os.listdir(images_paths)
    # masks_ids = os.listdir(masks_paths)

    images_ids = sorted(os.listdir(images_paths))
    masks_ids = sorted(os.listdir(masks_paths))

    X = np.zeros((len(images_ids), im_height, im_width, 3), dtype=np.float32)
    y = np.zeros((len(masks_ids), im_height, im_width, 1), dtype=np.float32)
    print('Getting and resizing images ... ')
    for n in range (len(images_ids)):
        # Load images
        img = img_to_array(load_img(os.path.join(images_paths,images_ids[n]), grayscale=False))
        x_img = resize(img, (im_height, im_width, 3), mode='constant', preserve_range=True)

        # Load masks
        mask = img_to_array(load_img(os.path.join(masks_paths,masks_ids[n]), grayscale=True))
        mask = resize(mask, (im_height, im_width, 1), mode='constant', preserve_range=True)

        # mask_vector = mask.reshape(mask.shape[0],mask.shape[1])
        # for ind in mask_vector:
        #   print(ind)

        # Save images
        X[n] = x_img / 255
        y[n] = mask / 255
        # mask = (mask > 0.5).astype(np.uint8)
        # y[n] = mask
        # print(mask)
        # print(np.max(mask))
    print('Done!')
    return X, y

# X, y = get_data(train_data)

In [ ]:
# import os
# import sys
# from PIL import Image
# import numpy as np

# def load_data(path, data_name):

#     print('loading '+data_name+' data...')
#     if data_name == 'dataset':
#         imgs_list = []
#         masks_list = []
#         items = os.listdir(os.path.join(path,'x'))
#         # items = sorted(it, key=lambda x: int(os.path.splitext(x)[0]))
#         for item in items:
#             imgs_list.append(np.array(Image.open(os.path.join(path,'x',item)).convert('RGB')))
#             masks_list.append(np.array(Image.open(os.path.join(path,'y',item)).convert('L')))
#             print(item)

#     imgs_np = np.asarray(imgs_list)
#     masks_np = np.asarray(masks_list)

#     x = np.asarray(imgs_np, dtype=np.float32)/255
#     y = np.asarray(masks_np, dtype=np.float32)/255

#     if len(y.shape) == 3:
#         y = y.reshape(y.shape[0], y.shape[1], y.shape[2], 1)
#         # x = x.reshape(x.shape[0], x.shape[1], x.shape[2], 3)

#     print("Successfully loaded data from "+path)
#     print("data shape:", x.shape,y.shape)

#     return x, y

In [ ]:
# data_name='dataset'

# X_train, y_train = load_data(train_data, data_name)
# X_valid, y_valid = load_data(valid_data, data_name)
# X, y = get_data(train_data)
# X_valid, y_valid = get_data(valid_data)

In [ ]:
# X,y = get_data(train_data)

from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.20, random_state=42)

# # from keras.models import load_model
# # model.load_weights('/content/gdrive/MyDrive/ISIC_2017_Swin/Weights_BCDU_Swin_Sep_Conv_DiceLoss_9827.h5')

In [ ]:
# from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
nb_epoch = 100
batch_size=4
Earlystop = EarlyStopping(monitor='val_loss', mode='min',patience=9, min_delta=0.001)
anne = ReduceLROnPlateau(monitor='val_loss', factor=0.25, patience=5, verbose=1, min_lr=1e-9)
checkpoint = ModelCheckpoint('/content/gdrive/MyDrive/ISIC_2017_Swin/Leather/Weights_Swin_Trans_Weights_Swin_Trans_Leather.h5', verbose=1, save_best_only=True,save_weights_only=True)
callbacks = [anne, checkpoint, Earlystop]

In [ ]:
# from keras.models import load_model
# model.load_weights('/content/gdrive/MyDrive/ISIC_2017_Swin/Weights_BCDU_Swin_Sep_Conv_DiceLoss_9827.h5')
from keras.models import load_model
model.load_weights('/content/gdrive/MyDrive/ISIC_2017_Swin/Weights_Swin_Trans_Weights_Swin_Trans_Brain_MRI.h5')

In [ ]:
results = model.fit(X_train, y_train, batch_size=batch_size, epochs=nb_epoch,callbacks=callbacks,
                    validation_data=(X_valid, y_valid))

In [ ]:
# from keras.models import load_model
# model.load_weights('/content/gdrive/MyDrive/ISIC_2017_Swin/Leather/Weights_Swin_Trans_Weights_Swin_Trans_Leather.h5')

In [ ]:
# valid_data = '/content/ISIC2018_256x256/test' #data path
X_test, y_test = get_data(valid_data)
# X_test = X_valid
# y_test = y_valid

In [ ]:
preds_test = model.predict(X_test, verbose=1)

In [ ]:
preds_test=np.array(preds_test)

In [ ]:
for i in range(preds_test.shape[0]):
  plt.imsave("/content/gdrive/MyDrive/ISIC_2017_Swin/segmentations/"+str(i+1)+".png", preds_test[i,:,:,0],cmap='gray') # binary segmenation
  plt.imsave("/content/gdrive/MyDrive/ISIC_2017_Swin/GT/"+str(i+1)+".tiff",y_test[i,:,:,0],cmap='gray') # binary segmenation

  # plt.imsave("/content/drive/MyDrive/Multiply/masks/"+str(i+1)+".png",y_test[i,:,:,0],cmap='gray') # binary segmenation

In [ ]:
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings("ignore")

def evaluate_metrics(y_test, y_pred):
    n = y_pred.shape[0]
    all_accuracy = np.zeros(n)
    all_dice = np.zeros(n)
    all_jaccard = np.zeros(n)
    all_sensitivity = np.zeros(n)
    all_specificity = np.zeros(n)
    for i in range(n):
        gt, pred = y_test[i], y_pred[i]
        gt_flt = np.ndarray.flatten(gt)
        pred_flt = np.ndarray.flatten(pred)

        precisions, recalls, thresholds = precision_recall_curve(gt_flt, pred_flt)
        f1 = 2*(precisions * recalls) / (precisions + recalls)
        max_value = np.argmax(f1)
        # precision, recall, thres = precisions[max_value], recalls[max_value], thresholds[max_value]
        thres = thresholds[max_value]
        # maxval = 255
        # pred_mask= (pred>= thres)
        pred_mask = (pred_flt >= thres)
        pred_label = pred_mask*1
        # pred_label = np.ndarray.flatten(pred_label)

        tn, fp, fn, tp = confusion_matrix(gt_flt, pred_label).ravel()

        accuracy = (tp + tn) / (tp + tn + fp + fn)
        iou = tp / (tp + fp + fn)
        dice = 2*tp / (2*tp + fp + fn)
        specificity = tn / (tn + fp)
        recall = tp / (tp + fn)

        all_accuracy[i] = accuracy
        all_dice[i] = dice
        all_jaccard[i] = iou
        all_sensitivity[i] = recall
        all_specificity[i] = specificity

    print('Accuracy: {:4f}, Dice: {:4f}, Jaccard: {:4f}, Sensitivity: {:4f}, Specificity: {:4f}'.format(
        np.nanmean(all_accuracy), np.nanmean(all_dice), np.nanmean(all_jaccard), np.nanmean(all_sensitivity), np.nanmean(all_specificity)
    ))
    return all_accuracy, all_dice, all_jaccard, all_sensitivity, all_specificity

In [ ]:
evl = evaluate_metrics(y_test, preds_test)